
# 02 | Data Exploration

**Objective:** Explore the 2026 FIFA World Cup dataset using SQL to understand match results, team participation, and tournament performance.

I use SQL queries to filter, sort, and summarize match data before moving into more advanced team and player analysis.



## 2.1 | Connecting to the Database

I connect to the SQLite database created in the previous notebook to begin exploring the tournament data.


In [1]:

import pandas as pd
import sqlite3
from pathlib import Path

# Connect to existing database
db_path = Path.cwd().parent / "Data" / "Prepared" / "world_cup_2026.db"

conn = sqlite3.connect(db_path)

print("Connected to World Cup database!")


Connected to World Cup database!



## 2.2 | Exploring Match Results

I retrieve a sample of match results to review the available information and practice basic SQL queries.


In [2]:

query = """
SELECT
    date,
    home_team,
    away_team,
    home_score,
    away_score
FROM matches
LIMIT 10;
"""

display(pd.read_sql_query(query, conn))


,date,home_team,away_team,home_score,away_score
0,11 June 2026,Mexico,South Africa,2,0
1,11 June 2026,Korea Republic,Czechia,2,1
2,12 June 2026,Canada,Bosnia and Herzegovina,1,1
3,12 June 2026,USA,Paraguay,4,1
4,13 June 2026,Haiti,Scotland,0,1
5,13 June 2026,Australia,Türkiye,2,0
6,13 June 2026,Brazil,Morocco,1,1
7,13 June 2026,Qatar,Switzerland,1,1
8,14 June 2026,Côte d'Ivoire,Ecuador,1,0
9,14 June 2026,Germany,Curaçao,7,1



## 2.3 | Analyzing Spain's Match Results

I filter the match dataset to examine Spain's results throughout the tournament, regardless of whether the team played at home or away.


In [3]:

# Retrieve all matches involving Spain
query = """
SELECT
    date,
    home_team,
    away_team,
    home_score,
    away_score,
    "group" AS stage
FROM matches
WHERE home_team = 'Spain'
   OR away_team = 'Spain'
ORDER BY match_number;
"""

display(pd.read_sql_query(query, conn))

,date,home_team,away_team,home_score,away_score,stage
0,15 June 2026,Spain,Cabo Verde,0,0,Group H
1,21 June 2026,Spain,Saudi Arabia,4,0,Group H
2,26 June 2026,Uruguay,Spain,0,1,Group H
3,02 July 2026,Spain,Austria,3,0,Round of 32
4,06 July 2026,Portugal,Spain,0,1,Round of 16
5,10 July 2026,Spain,Belgium,2,1,Quarter-ﬁnal
6,14 July 2026,France,Spain,0,2,Semi-ﬁnal
7,19 July 2026,Spain,Argentina,1,0,Final



## 2.4 | Calculating Spain's Performance

I use SQL aggregations and conditional statements to calculate Spain's total matches, wins, draws, losses, and goals scored and conceded.


In [4]:

# Calculate Spain's tournament statistics
query = """
SELECT
    COUNT(*) AS matches_played,

    SUM(CASE
        WHEN home_team = 'Spain' AND home_score > away_score THEN 1
        WHEN away_team = 'Spain' AND away_score > home_score THEN 1
        ELSE 0
    END) AS wins,

    SUM(CASE
        WHEN home_score = away_score THEN 1
        ELSE 0
    END) AS draws,

    SUM(CASE
        WHEN home_team = 'Spain' AND home_score < away_score THEN 1
        WHEN away_team = 'Spain' AND away_score < home_score THEN 1
        ELSE 0
    END) AS losses,

    SUM(CASE
        WHEN home_team = 'Spain' THEN home_score
        ELSE away_score
    END) AS goals_scored,

    SUM(CASE
        WHEN home_team = 'Spain' THEN away_score
        ELSE home_score
    END) AS goals_conceded

FROM matches
WHERE home_team = 'Spain'
   OR away_team = 'Spain';
"""

display(pd.read_sql_query(query, conn))


,matches_played,wins,draws,losses,goals_scored,goals_conceded
0,8,7,1,0,14,1



## 2.5 | Comparing Spain and Argentina

I compare Spain and Argentina using SQL to evaluate their match results, scoring performance, and defensive records.


In [5]:

# Compare Spain and Argentina
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
)

SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(CASE WHEN goals_for > goals_against THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN goals_for = goals_against THEN 1 ELSE 0 END) AS draws,
    SUM(CASE WHEN goals_for < goals_against THEN 1 ELSE 0 END) AS losses,
    SUM(goals_for) AS goals_scored,
    SUM(goals_against) AS goals_conceded,
    SUM(goals_for - goals_against) AS goal_difference
FROM team_matches
WHERE team IN ('Spain', 'Argentina')
GROUP BY team
ORDER BY wins DESC;
"""

display(pd.read_sql_query(query, conn))


,team,matches_played,wins,draws,losses,goals_scored,goals_conceded,goal_difference
0,Spain,8,7,1,0,14,1,13
1,Argentina,8,7,0,1,19,8,11



## 2.6 | Ranking Team Performance

I expand the analysis to all participating teams and rank their performance using wins, goal difference, and total goals scored.


In [6]:

# Rank teams by tournament performance
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
)

SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(CASE WHEN goals_for > goals_against THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN goals_for = goals_against THEN 1 ELSE 0 END) AS draws,
    SUM(CASE WHEN goals_for < goals_against THEN 1 ELSE 0 END) AS losses,
    SUM(goals_for) AS goals_scored,
    SUM(goals_against) AS goals_conceded,
    SUM(goals_for - goals_against) AS goal_difference
FROM team_matches
GROUP BY team
ORDER BY wins DESC, goal_difference DESC, goals_scored DESC;
"""

team_rankings = pd.read_sql_query(query, conn)

display(team_rankings.head(15))


,team,matches_played,wins,draws,losses,goals_scored,goals_conceded,goal_difference
0,Spain,8,7,1,0,14,1,13
1,Argentina,8,7,0,1,19,8,11
2,France,8,6,0,2,20,10,10
3,England,8,6,1,1,20,12,8
4,Mexico,5,4,0,1,10,3,7
5,Norway,6,4,0,2,13,11,2
6,Belgium,6,3,2,1,14,7,7
7,Brazil,5,3,1,1,10,4,6
8,Morocco,6,3,2,1,10,6,4
9,Switzerland,6,3,2,1,10,6,4



## 2.7 | Evaluating Scoring Efficiency

I calculate average goals scored and conceded per match to compare attacking productivity and defensive performance across teams.


In [7]:

# Calculate scoring and defensive averages
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
)

SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(goals_for) AS total_goals,
    ROUND(AVG(goals_for), 2) AS goals_per_match,
    ROUND(AVG(goals_against), 2) AS conceded_per_match,
    ROUND(AVG(goals_for - goals_against), 2) AS avg_goal_difference
FROM team_matches
GROUP BY team
HAVING COUNT(*) >= 4
ORDER BY goals_per_match DESC;
"""

display(pd.read_sql_query(query, conn))


,team,matches_played,total_goals,goals_per_match,conceded_per_match,avg_goal_difference
0,Netherlands,4,11,2.75,1.25,1.50
1,Germany,4,11,2.75,1.25,1.50
2,Senegal,4,10,2.50,2.25,0.25
3,France,8,20,2.50,1.25,1.25
4,England,8,20,2.50,1.50,1.00
5,Argentina,8,19,2.38,1.00,1.38
6,Belgium,6,14,2.33,1.17,1.17
7,USA,5,11,2.20,1.60,0.60
8,Norway,6,13,2.17,1.83,0.33
9,Mexico,5,10,2.00,0.60,1.40
